# 07. k-Nearest Neighbors

Distance-based majority vote. Scaling matters (default preprocess).
Tune `n_neighbors` (k) with stratified CV (F1).

In [1]:
import sys
sys.path.append("..")

from sklearn.neighbors import KNeighborsClassifier

from src.data import load_adult
from src.preprocess import split_features_target, make_model_pipeline
from src.evaluate import cross_val_scores, evaluate_on_test, save_results

X, y = split_features_target(load_adult("train"))
X_test, y_test = split_features_target(load_adult("test"))
print(X.shape, y.mean())

(32561, 12) 0.2408095574460244


In [2]:
candidates = [5, 11, 15, 25, 35]
selection_metric = "f1"

cv_by_k = {}
best_k = None
best_score = -1.0

for k in candidates:
    model = KNeighborsClassifier(n_neighbors=k, n_jobs=-1)
    pipe = make_model_pipeline(model)
    cv = cross_val_scores(pipe, X, y)
    cv_by_k[k] = cv
    score = cv[selection_metric]["mean"]
    print(f"k={k} CV {selection_metric}={score:.4f} (±{cv[selection_metric]['std']:.4f})")

    if score > best_score:
        best_score = score
        best_k = k

print("selected k:", best_k)

k=5 CV f1=0.6406 (±0.0087)
k=11 CV f1=0.6584 (±0.0065)
k=15 CV f1=0.6624 (±0.0055)
k=25 CV f1=0.6636 (±0.0073)
k=35 CV f1=0.6698 (±0.0057)
selected k: 35


In [3]:
final_model = KNeighborsClassifier(n_neighbors=best_k, n_jobs=-1)

cv = cv_by_k[best_k]
pipe2 = make_model_pipeline(final_model)
test = evaluate_on_test(pipe2, X, y, X_test, y_test)

path = save_results(
    f"knn_k{best_k}",
    cv,
    test,
    extra={
        "n_neighbors": best_k,
        "candidates": candidates,
        "selection_metric": selection_metric,
        "cv_by_k": {
            str(k): {m: v["mean"] for m, v in summary.items()}
            for k, summary in cv_by_k.items()
        },
    },
)

print("Test:", test["metrics"])
print("Confusion matrix:", test["confusion_matrix"])
print("saved:", path)

Test: {'accuracy': 0.8480437319574965, 'balanced_accuracy': 0.7748081599982938, 'f1': 0.6641325006787945, 'roc_auc': 0.8984013594560671, 'average_precision': 0.7339882053227808}
Confusion matrix: [[11361, 1074], [1400, 2446]]
saved: /Users/kang/Desktop/ML project/Estimating-Income/results/knn_k35.json
